In [389]:
import json 
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import fbeta_score

### Load Data

In [390]:
with open("/Users/christophlaute/Thesis/DuplicateDetectionThesis/datasets/FIVR-200K/dataset/annotation.json") as f:
    annotations = json.load(f)


In [391]:
def load_detections(detections_path):
    with open(detections_path, "r") as f:
        data = json.load(f)
    return data["detections"]

In [392]:
def load_similarities(similarities_path):
    with open(similarities_path, "r") as f:
        data = json.load(f)
    return data["similarities"]

metrics

In [393]:
def calculate_groundtruth(query_id, relevant_labels, annotations):
    gt = set()
    for label in relevant_labels:
        gt.update(annotations.get(query_id, {}).get(label, []))
    return gt


In [394]:
def compute_mean_average(similarities, 
                         relevant_labels, annotations):
    """compute mean average precision for each query and return a dictionary with the results"""
    per_query_ap = {}
    for query_id, reference_ids in similarities.items():
        gt = calculate_groundtruth(query_id, relevant_labels, annotations)
        gt = gt & set(reference_ids.keys())
        if not gt:
            continue

        ranked = sorted(reference_ids.items(),key=lambda x: x[1], reverse=True)
        relevants =0
        precisions = []
        for i, (ref_id, sim) in enumerate(ranked):
            if ref_id == query_id:
                continue
            if ref_id in gt:
                relevants += 1
                precisions.append(relevants / (i + 1))

        per_query_ap[query_id] = np.sum(precisions)/len(gt)
    return per_query_ap
            

            


        

In [395]:
def compute_fbeta(detections, relevant_labels, annotations):
    nd_only = list(relevant_labels) == ["ND"]
    per_query_fbeta = {}
    for query_id, reference_ids in detections.items():
        gt = calculate_groundtruth(query_id, relevant_labels, annotations)
        if not gt:
            continue
        exclude = set(annotations.get(query_id, {}).get("DS", [])) if nd_only else set()
        y_true, y_pred = [], []
        for ref_id, predicted in reference_ids.items():
            if ref_id == query_id:
                continue
            if ref_id in exclude:
                continue
            y_true.append(1 if ref_id in gt else 0)
            y_pred.append(1 if predicted else 0)
        per_query_fbeta[query_id] = fbeta_score(y_true, y_pred, beta=0.5, zero_division=0)
    return per_query_fbeta

In [396]:

def get_detection_pairs(detections, relevant_labels, annotations):
    nd_only = list(relevant_labels) == ["ND"]
    pairs = {}
    for query_id, reference_ids in detections.items():
        gt = calculate_groundtruth(query_id, relevant_labels, annotations)
        if not gt:
            continue
        exclude = set(annotations.get(query_id, {}).get("DS", [])) if nd_only else set()
        y_true, y_pred = [], []
        for ref_id, predicted in reference_ids.items():
            if ref_id == query_id:
                continue
            if ref_id in exclude:
                continue
            y_true.append(1 if ref_id in gt else 0)
            y_pred.append(1 if predicted else 0)
        if y_true:
            pairs[query_id] = (np.array(y_true), np.array(y_pred))
    return pairs

def _fbeta_from_counts(tp, fp, fn, beta=0.5):
    if tp == 0:
        return 0.0
    b2 = beta * beta
    prec = tp / (tp + fp)
    rec  = tp / (tp + fn)
    return (1 + b2) * prec * rec / (b2 * prec + rec)

def _counts(pairs):
        tps = np.array([int(np.sum((pairs[q][0]==1) & (pairs[q][1]==1))) for q in queries])
        fps = np.array([int(np.sum((pairs[q][0]==0) & (pairs[q][1]==1))) for q in queries])
        fns = np.array([int(np.sum((pairs[q][0]==1) & (pairs[q][1]==0))) for q in queries])
        return tps, fps, fns

def pooled_paired_bootstrap_test(pairs_a, pairs_b, n_bootstrap=10000, seed=42):
    rng = np.random.RandomState(seed)
    queries = sorted(set(pairs_a) & set(pairs_b))
    n = len(queries)

    

    tpa, fpa, fna = _counts(pairs_a)
    tpb, fpb, fnb = _counts(pairs_b)

    orig_diff = (_fbeta_from_counts(tpa.sum(), fpa.sum(), fna.sum())
               - _fbeta_from_counts(tpb.sum(), fpb.sum(), fnb.sum()))

    diffs = np.empty(n_bootstrap)
    for i in range(n_bootstrap):
        c = np.bincount(rng.choice(n, n, replace=True), minlength=n)
        da = _fbeta_from_counts(c @ tpa, c @ fpa, c @ fna)
        db = _fbeta_from_counts(c @ tpb, c @ fpb, c @ fnb)
        diffs[i] = da - db

    p = np.mean(diffs <= 0) if orig_diff > 0 else np.mean(diffs >= 0)
    ci = np.percentile(diffs, [2.5, 97.5])
    return orig_diff, ci, p

def run_pooled_detection_bootstrap(experiments_pairs):
    """Run all pairwise pooled bootstrap tests."""
    for exp_name, methods_pairs in experiments_pairs.items():
        print(f"Experiment: {exp_name}")
        methods = list(methods_pairs.keys())
        for i in range(len(methods)):
            for j in range(i+1, len(methods)):
                a, b = methods[i], methods[j]
                diff, ci, p = pooled_paired_bootstrap_test(methods_pairs[a], methods_pairs[b])
                pstr = "<0.0001" if p == 0 else f"{p:.4f}"
                print(f"  {a} vs {b}: diff={diff:+.4f} CI=[{ci[0]:+.4f},{ci[1]:+.4f}] p={pstr}")
        print()


In [397]:
with open("/Users/christophlaute/Thesis/DuplicateDetectionThesis/CompactSIamese/fivr_subset_50k/fivr5k_vid.json") as f:
    _fivr5k_ids = json.load(f)
db5k_stems = {Path(v).stem for v in _fivr5k_ids["database"]}

def filter_to_db(data, valid_refs):
    return {q: {r: v for r, v in refs.items() if r in valid_refs} for q, refs in data.items()}

 main experiments

In [398]:
#main comparison --5k -- retrieval
relevant_labels = ["ND", "DS"]
hap_retrieval_5k = load_similarities("similarities_hap_5k_all.json")
hap_retrieval_5k_map = compute_mean_average(hap_retrieval_5k, relevant_labels, annotations)

line_retrieval_5k = filter_to_db(load_similarities("similarities_line_fivr5k_all_full.json"), db5k_stems)
line_retrieval_5k_map = compute_mean_average(line_retrieval_5k, relevant_labels    , annotations)
vvs_retrieval_5k = load_similarities("similarities_vvs_fivr5k_all.json")
vvs_retrieval_5k_map = compute_mean_average(vvs_retrieval_5k, relevant_labels, annotations)
tmk_retrieval_5k = load_similarities("similarities_tmk_fivr5k_all.json")
tmk_retrieval_5k_map = compute_mean_average(tmk_retrieval_5k, relevant_labels, annotations)
siamese_retrieval_5k = load_similarities("similarities_siamese_5k_all.json")
siamese_retrieval_5k_map = compute_mean_average(siamese_retrieval_5k, relevant_labels, annotations)
main_retrieval_5k_pd = pd.DataFrame({
    "hap": hap_retrieval_5k_map,
    "line": line_retrieval_5k_map,
    "vvs": vvs_retrieval_5k_map,
    "tmk": tmk_retrieval_5k_map,
    "siamese": siamese_retrieval_5k_map,})


In [399]:
#main comparison --50k -- retrieval
relevant_labels = ["ND", "DS"]
hap_retrieval_50k = load_similarities("similarities_hap_50k_all.json")
hap_retrieval_50k_map = compute_mean_average(hap_retrieval_50k, relevant_labels, annotations)
line_retrieval_50k = load_similarities("similarities_line_fivr50k_all_full.json")
line_retrieval_50k_map = compute_mean_average(line_retrieval_50k, relevant_labels, annotations)
vvs_retrieval_50k = load_similarities("similarities_vvs_fivr50k_all.json")
vvs_retrieval_50k_map = compute_mean_average(vvs_retrieval_50k, relevant_labels, annotations)
tmk_retrieval_50k = load_similarities("similarities_tmk_fivr50k_all.json")
tmk_retrieval_50k_map = compute_mean_average(tmk_retrieval_50k, relevant_labels, annotations)
siamese_retrieval_50k = load_similarities("similarities_siamese_50k_all.json")
siamese_retrieval_50k_map = compute_mean_average(siamese_retrieval_50k, relevant_labels, annotations)
main_retrieval_50k_pd = pd.DataFrame({
    "hap": hap_retrieval_50k_map,
    "line": line_retrieval_50k_map,
    "vvs": vvs_retrieval_50k_map,
    "tmk": tmk_retrieval_50k_map,
    "siamese": siamese_retrieval_50k_map,})

In [400]:
#main comparison --5k -- detection - nd+ds
relevant_labels = ["ND", "DS"]
hap_detection_5k = load_detections("similarities_hap_5k_all.json")
hap_detection_5k_fbeta = compute_fbeta(hap_detection_5k, relevant_labels, annotations)    
line_detection_5k = filter_to_db(load_detections("similarities_line_fivr5k_all_full.json"), db5k_stems)
line_detection_5k_fbeta = compute_fbeta(line_detection_5k, relevant_labels, annotations)
vvs_detection_5k = load_detections("similarities_vvs_fivr5k_all.json")
vvs_detection_5k_fbeta = compute_fbeta(vvs_detection_5k, relevant_labels , annotations)
tmk_detection_5k = load_detections("similarities_tmk_fivr5k_all.json")
tmk_detection_5k_fbeta = compute_fbeta(tmk_detection_5k, relevant_labels, annotations)
siamese_detection_5k = load_detections("similarities_siamese_5k_all.json")
siamese_detection_5k_fbeta = compute_fbeta(siamese_detection_5k, relevant_labels, annotations)
main_detection_5k_pd = pd.DataFrame({
    "hap": hap_detection_5k_fbeta,
    "line": line_detection_5k_fbeta,
    "vvs": vvs_detection_5k_fbeta,
    "tmk": tmk_detection_5k_fbeta,
    "siamese": siamese_detection_5k_fbeta,})

In [401]:
#main comparison --5k -- detection - nd
relevant_labels = ["ND"]
hap_detection_5k_nd = load_detections("similarities_hap_5k_nd.json")    
hap_detection_5k_nd_fbeta = compute_fbeta(hap_detection_5k_nd, relevant_labels, annotations)
line_detection_5k_nd = filter_to_db(load_detections("similarities_line_fivr5k_nd_full.json"), db5k_stems)
line_detection_5k_nd_fbeta = compute_fbeta(line_detection_5k_nd, relevant_labels, annotations)

vvs_detection_5k_nd = load_detections("similarities_vvs_fivr5k_nd.json")
vvs_detection_5k_nd_fbeta = compute_fbeta(vvs_detection_5k_nd, relevant_labels, annotations)
tmk_detection_5k_nd = load_detections("similarities_tmk_fivr5k_nd.json")
tmk_detection_5k_nd_fbeta = compute_fbeta(tmk_detection_5k_nd, relevant_labels, annotations)
siamese_detection_5k_nd = load_detections("similarities_siamese_5k_nd.json")
siamese_detection_5k_nd_fbeta = compute_fbeta(siamese_detection_5k_nd, relevant_labels, annotations)
main_detection_5k_nd_pd = pd.DataFrame({
    "hap": hap_detection_5k_nd_fbeta,
    "line": line_detection_5k_nd_fbeta,
    "vvs": vvs_detection_5k_nd_fbeta,
    "tmk": tmk_detection_5k_nd_fbeta,
    "siamese": siamese_detection_5k_nd_fbeta,})

In [402]:
#main comparison --50k -- detection - nd+ds
relevant_labels = ["ND", "DS"]
hap_detection_50k = load_detections("similarities_hap_50k_all.json")
hap_detection_50k_fbeta = compute_fbeta(hap_detection_50k, relevant_labels, annotations)    
line_detection_50k = load_detections("similarities_line_fivr50k_all_full.json")
line_detection_50k_fbeta = compute_fbeta(line_detection_50k, relevant_labels, annotations)
vvs_detection_50k = load_detections("similarities_vvs_fivr50k_all.json")
vvs_detection_50k_fbeta = compute_fbeta(vvs_detection_50k, relevant_labels    , annotations)
tmk_detection_50k = load_detections("similarities_tmk_fivr50k_all.json")
tmk_detection_50k_fbeta = compute_fbeta(tmk_detection_50k, relevant_labels, annotations)
siamese_detection_50k = load_detections("similarities_siamese_50k_all.json")
siamese_detection_50k_fbeta = compute_fbeta(siamese_detection_50k, relevant_labels, annotations)
main_detection_50k_pd = pd.DataFrame({
    "hap": hap_detection_50k_fbeta,
    "line": line_detection_50k_fbeta,
    "vvs": vvs_detection_50k_fbeta,
    "tmk": tmk_detection_50k_fbeta,
    "siamese": siamese_detection_50k_fbeta,})

In [403]:
#main comparison --50k -- detection - nd
relevant_labels = ["ND"]
hap_detection_50k_nd = load_detections("similarities_hap_50k_nd.json")    
hap_detection_50k_nd_fbeta = compute_fbeta(hap_detection_50k_nd, relevant_labels, annotations)
line_detection_50k_nd = load_detections("similarities_line_fivr50k_nd_full.json")
line_detection_50k_nd_fbeta = compute_fbeta(line_detection_50k_nd, relevant_labels, annotations)

vvs_detection_50k_nd = load_detections("similarities_vvs_fivr50k_nd.json")
vvs_detection_50k_nd_fbeta = compute_fbeta(vvs_detection_50k_nd, relevant_labels, annotations)
tmk_detection_50k_nd = load_detections("similarities_tmk_fivr50k_nd.json")
tmk_detection_50k_nd_fbeta = compute_fbeta(tmk_detection_50k_nd, relevant_labels, annotations)
siamese_detection_50k_nd = load_detections("similarities_siamese_50k_nd.json")
siamese_detection_50k_nd_fbeta = compute_fbeta(siamese_detection_50k_nd, relevant_labels, annotations)
main_detection_50k_nd_pd = pd.DataFrame({
    "hap": hap_detection_50k_nd_fbeta,
    "line": line_detection_50k_nd_fbeta,
    "vvs": vvs_detection_50k_nd_fbeta,
    "tmk": tmk_detection_50k_nd_fbeta,
    "siamese": siamese_detection_50k_nd_fbeta,})

siamese ablations

In [ ]:
# HAP-DPP frame selection experiment 
relevant_labels = ["ND", "DS"]
hap_full_all = load_detections("similarities_hap_50k_all.json")
hap_full_all_fbeta = compute_fbeta(hap_full_all, relevant_labels, annotations)
hap_dpp_uniform_all = load_detections("similarities_hap_dpp_50k_all_uniform_k2.json")
hap_dpp_uniform_all_fbeta = compute_fbeta(hap_dpp_uniform_all, relevant_labels, annotations)
hap_dpp_interframe_all = load_detections("similarities_hap_dpp_50k_all_interframe_diff_k2.json")
hap_dpp_interframe_all_fbeta = compute_fbeta(hap_dpp_interframe_all, relevant_labels, annotations)
hap_dpp_dpp_all = load_detections("similarities_hap_dpp_50k_all_dpp_k2.json")
hap_dpp_dpp_all_fbeta = compute_fbeta(hap_dpp_dpp_all, relevant_labels, annotations)
hap_dpp_detection_all_pd = pd.DataFrame({
    "full": hap_full_all_fbeta,
    "uniform": hap_dpp_uniform_all_fbeta,
    "interframe_diff": hap_dpp_interframe_all_fbeta,
    "dpp": hap_dpp_dpp_all_fbeta,})

relevant_labels = ["ND"]
hap_full_nd = load_detections("similarities_hap_50k_nd.json")
hap_full_nd_fbeta = compute_fbeta(hap_full_nd, relevant_labels, annotations)
hap_dpp_uniform_nd = load_detections("similarities_hap_dpp_50k_nd_uniform_k2.json")
hap_dpp_uniform_nd_fbeta = compute_fbeta(hap_dpp_uniform_nd, relevant_labels, annotations)
hap_dpp_interframe_nd = load_detections("similarities_hap_dpp_50k_nd_interframe_diff_k2.json")
hap_dpp_interframe_nd_fbeta = compute_fbeta(hap_dpp_interframe_nd, relevant_labels, annotations)
hap_dpp_dpp_nd = load_detections("similarities_hap_dpp_50k_nd_dpp_k2.json")
hap_dpp_dpp_nd_fbeta = compute_fbeta(hap_dpp_dpp_nd, relevant_labels, annotations)
hap_dpp_detection_nd_pd = pd.DataFrame({
    "full": hap_full_nd_fbeta,
    "uniform": hap_dpp_uniform_nd_fbeta,
    "interframe_diff": hap_dpp_interframe_nd_fbeta,
    "dpp": hap_dpp_dpp_nd_fbeta,})

In [405]:
# HAP-DPP frame selection experiment (50K retrieval mAP)
relevant_labels = ["ND", "DS"]
hap_full_retr = load_similarities("similarities_hap_50k_all.json")
hap_full_retr_map = compute_mean_average(hap_full_retr, relevant_labels, annotations)
hap_dpp_uniform_retr = load_similarities("similarities_hap_dpp_50k_all_uniform_k2.json")
hap_dpp_uniform_retr_map = compute_mean_average(hap_dpp_uniform_retr, relevant_labels, annotations)
hap_dpp_interframe_retr = load_similarities("similarities_hap_dpp_50k_all_interframe_diff_k2.json")
hap_dpp_interframe_retr_map = compute_mean_average(hap_dpp_interframe_retr, relevant_labels, annotations)
hap_dpp_dpp_retr = load_similarities("similarities_hap_dpp_50k_all_dpp_k2.json")
hap_dpp_dpp_retr_map = compute_mean_average(hap_dpp_dpp_retr, relevant_labels, annotations)
hap_dpp_retrieval_pd = pd.DataFrame({
    "full": hap_full_retr_map,
    "uniform": hap_dpp_uniform_retr_map,
    "interframe_diff": hap_dpp_interframe_retr_map,
    "dpp": hap_dpp_dpp_retr_map,})

In [406]:

hap_dpp_vs_all_50k_pd = pd.DataFrame({
    "hap":     hap_detection_50k_fbeta,
    "line":    line_detection_50k_fbeta,
    "vvs":     vvs_detection_50k_fbeta,
    "tmk":     tmk_detection_50k_fbeta,
    "siamese": siamese_detection_50k_fbeta,
    "hap_dpp": hap_dpp_dpp_all_fbeta,})

hap_dpp_vs_all_50k_nd_pd = pd.DataFrame({
    "hap":     hap_detection_50k_nd_fbeta,
    "line":    line_detection_50k_nd_fbeta,
    "vvs":     vvs_detection_50k_nd_fbeta,
    "tmk":     tmk_detection_50k_nd_fbeta,
    "siamese": siamese_detection_50k_nd_fbeta,
    "hap_dpp": hap_dpp_dpp_nd_fbeta,})

In [407]:

relevant_labels = ["ND", "DS"]
siam_chamfer = load_similarities("results_siamese_retrieval_5k_chamfer_siamese.json")
siam_chamfer_map = compute_mean_average(siam_chamfer, relevant_labels, annotations)
siam_pred_max = load_similarities("results_siamese_retrieval_5k_pred_max_siamese.json")
siam_pred_max_map = compute_mean_average(siam_pred_max, relevant_labels, annotations)
siam_pred_chamfer = load_similarities("results_siamese_retrieval_5k_pred_chamfer_siamese.json")
siam_pred_chamfer_map = compute_mean_average(siam_pred_chamfer, relevant_labels, annotations)
siam_ablation_5k_pd = pd.DataFrame({
    "chamfer": siam_chamfer_map,
    "pred_max": siam_pred_max_map,
    "pred_chamfer": siam_pred_chamfer_map,})


In [408]:

relevant_labels = ["ND", "DS"]
siam_baseline = load_similarities("results_siamese_retrieval_5k_pred_max_siamese.json")
siam_baseline_map = compute_mean_average(siam_baseline, relevant_labels, annotations)
siam_T10 = load_similarities("results_siamese_retrieval_5k_pred_max_T10_32x32_siamese.json")
siam_T10_map = compute_mean_average(siam_T10, relevant_labels, annotations)
siam_56x56 = load_similarities("results_siamese_retrieval_5k_pred_max_T15_56x56_siamese.json")
siam_56x56_map = compute_mean_average(siam_56x56, relevant_labels, annotations)
siam_preprocess_ablation_5k_pd = pd.DataFrame({
    "baseline": siam_baseline_map,
    "T10": siam_T10_map,
    "res56x56": siam_56x56_map,})

Line Ablations

In [409]:

relevant_labels = ["ND", "DS"]
line_full = load_similarities("similarities_line_fivr50k_all_full.json")
line_full_map = compute_mean_average(line_full, relevant_labels, annotations)
line_nosn = load_similarities("similarities_line_fivr50k_all_nosn.json")
line_nosn_map = compute_mean_average(line_nosn, relevant_labels, annotations)
line_nocw = load_similarities("similarities_line_fivr50k_all_nocw.json")
line_nocw_map = compute_mean_average(line_nocw, relevant_labels, annotations)
line_notc = load_similarities("similarities_line_fivr50k_all_notc.json")
line_notc_map = compute_mean_average(line_notc, relevant_labels, annotations)
line_notta = load_similarities("similarities_line_fivr50k_all_notta.json")
line_notta_map = compute_mean_average(line_notta, relevant_labels, annotations)
line_avg = load_similarities("similarities_line_fivr50k_all_avg.json")
line_avg_map = compute_mean_average(line_avg, relevant_labels, annotations)
line_ablation_50k_retr_pd = pd.DataFrame({
    "full": line_full_map,
    "nosn": line_nosn_map,
    "nocw": line_nocw_map,
    "notc": line_notc_map,
    "notta": line_notta_map,
    "avg": line_avg_map,})

In [410]:

relevant_labels = ["ND", "DS"]
line_full = filter_to_db(load_similarities("similarities_line_fivr5k_all_full.json"), db5k_stems)
line_full_map = compute_mean_average(line_full, relevant_labels, annotations)
line_nosn = filter_to_db(load_similarities("similarities_line_fivr5k_all_nosn.json"), db5k_stems)
line_nosn_map = compute_mean_average(line_nosn, relevant_labels, annotations)
line_nocw = filter_to_db(load_similarities("similarities_line_fivr5k_all_nocw.json"), db5k_stems)
line_nocw_map = compute_mean_average(line_nocw, relevant_labels, annotations)
line_notc = filter_to_db(load_similarities("similarities_line_fivr5k_all_notc.json"), db5k_stems)
line_notc_map = compute_mean_average(line_notc, relevant_labels, annotations)
line_notta = filter_to_db(load_similarities("similarities_line_fivr5k_all_notta.json"), db5k_stems)
line_notta_map = compute_mean_average(line_notta, relevant_labels, annotations)
line_avg = filter_to_db(load_similarities("similarities_line_fivr5k_all_avg.json"), db5k_stems)
line_avg_map = compute_mean_average(line_avg, relevant_labels, annotations)
line_ablation_5k_retr_pd = pd.DataFrame({
    "full": line_full_map,
    "nosn": line_nosn_map,
    "nocw": line_nocw_map,
    "notc": line_notc_map,
    "notta": line_notta_map,
    "avg": line_avg_map,})

In [411]:

relevant_labels = ["ND", "DS"]
line_full = filter_to_db(load_detections("similarities_line_fivr5k_all_full.json"), db5k_stems)
line_full_fbeta = compute_fbeta(line_full, relevant_labels, annotations)
line_nosn = filter_to_db(load_detections("similarities_line_fivr5k_all_nosn.json"), db5k_stems)
line_nosn_fbeta = compute_fbeta(line_nosn, relevant_labels, annotations)
line_nocw = filter_to_db(load_detections("similarities_line_fivr5k_all_nocw.json"), db5k_stems)
line_nocw_fbeta = compute_fbeta(line_nocw, relevant_labels, annotations)
line_notc = filter_to_db(load_detections("similarities_line_fivr5k_all_notc.json"), db5k_stems)
line_notc_fbeta = compute_fbeta(line_notc, relevant_labels, annotations)
line_notta = filter_to_db(load_detections("similarities_line_fivr5k_all_notta.json"), db5k_stems)
line_notta_fbeta = compute_fbeta(line_notta, relevant_labels, annotations)
line_avg = filter_to_db(load_detections("similarities_line_fivr5k_all_avg.json"), db5k_stems)
line_avg_fbeta = compute_fbeta(line_avg, relevant_labels, annotations)
line_ablation_5k_det_all_pd = pd.DataFrame({
    "full": line_full_fbeta,
    "nosn": line_nosn_fbeta,
    "nocw": line_nocw_fbeta,
    "notc": line_notc_fbeta,
    "notta": line_notta_fbeta,
    "avg": line_avg_fbeta,})

In [412]:

relevant_labels = ["ND", "DS"]
line_full = load_detections("similarities_line_fivr50k_all_full.json")
line_full_fbeta = compute_fbeta(line_full, relevant_labels, annotations)
line_nosn = load_detections("similarities_line_fivr50k_all_nosn.json")
line_nosn_fbeta = compute_fbeta(line_nosn, relevant_labels, annotations)
line_nocw = load_detections("similarities_line_fivr50k_all_nocw.json")
line_nocw_fbeta = compute_fbeta(line_nocw, relevant_labels, annotations)
line_notc = load_detections("similarities_line_fivr50k_all_notc.json")
line_notc_fbeta = compute_fbeta(line_notc, relevant_labels, annotations)
line_notta = load_detections("similarities_line_fivr50k_all_notta.json")
line_notta_fbeta = compute_fbeta(line_notta, relevant_labels, annotations)
line_avg = load_detections("similarities_line_fivr50k_all_avg.json")
line_avg_fbeta = compute_fbeta(line_avg, relevant_labels, annotations)
line_ablation_50k_det_all_pd = pd.DataFrame({
    "full": line_full_fbeta,
    "nosn": line_nosn_fbeta,
    "nocw": line_nocw_fbeta,
    "notc": line_notc_fbeta,
    "notta": line_notta_fbeta,
    "avg": line_avg_fbeta,})

In [413]:

relevant_labels = ["ND"]
line_full = filter_to_db(load_detections("similarities_line_fivr5k_all_full.json"), db5k_stems)
line_full_fbeta = compute_fbeta(line_full, relevant_labels, annotations)
line_nosn = filter_to_db(load_detections("similarities_line_fivr5k_all_nosn.json"), db5k_stems)
line_nosn_fbeta = compute_fbeta(line_nosn, relevant_labels, annotations)
line_nocw = filter_to_db(load_detections("similarities_line_fivr5k_all_nocw.json"), db5k_stems)
line_nocw_fbeta = compute_fbeta(line_nocw, relevant_labels, annotations)
line_notc = filter_to_db(load_detections("similarities_line_fivr5k_all_notc.json"), db5k_stems)
line_notc_fbeta = compute_fbeta(line_notc, relevant_labels, annotations)
line_notta = filter_to_db(load_detections("similarities_line_fivr5k_all_notta.json"), db5k_stems)
line_notta_fbeta = compute_fbeta(line_notta, relevant_labels, annotations)
line_avg = filter_to_db(load_detections("similarities_line_fivr5k_all_avg.json"), db5k_stems)
line_avg_fbeta = compute_fbeta(line_avg, relevant_labels, annotations)
line_ablation_5k_det_nd_pd = pd.DataFrame({
    "full": line_full_fbeta,
    "nosn": line_nosn_fbeta,
    "nocw": line_nocw_fbeta,
    "notc": line_notc_fbeta,
    "notta": line_notta_fbeta,
    "avg": line_avg_fbeta,})

In [414]:

relevant_labels = ["ND"]
line_full = load_detections("similarities_line_fivr50k_all_full.json")
line_full_fbeta = compute_fbeta(line_full, relevant_labels, annotations)
line_nosn = load_detections("similarities_line_fivr50k_all_nosn.json")
line_nosn_fbeta = compute_fbeta(line_nosn, relevant_labels, annotations)
line_nocw = load_detections("similarities_line_fivr50k_all_nocw.json")
line_nocw_fbeta = compute_fbeta(line_nocw, relevant_labels, annotations)
line_notc = load_detections("similarities_line_fivr50k_all_notc.json")
line_notc_fbeta = compute_fbeta(line_notc, relevant_labels, annotations)
line_notta = load_detections("similarities_line_fivr50k_all_notta.json")
line_notta_fbeta = compute_fbeta(line_notta, relevant_labels, annotations)
line_avg = load_detections("similarities_line_fivr50k_all_avg.json")
line_avg_fbeta = compute_fbeta(line_avg, relevant_labels, annotations)
line_ablation_50k_det_nd_pd = pd.DataFrame({
    "full": line_full_fbeta,
    "nosn": line_nosn_fbeta,
    "nocw": line_nocw_fbeta,
    "notc": line_notc_fbeta,
    "notta": line_notta_fbeta,
    "avg": line_avg_fbeta,})

In [415]:
all_dfs = {
    "main_retrieval_5k": main_retrieval_5k_pd,
    "main_retrieval_50k": main_retrieval_50k_pd,
    "main_detection_5k": main_detection_5k_pd,
    "main_detection_50k": main_detection_50k_pd,
    "main_detection_5k_nd": main_detection_5k_nd_pd,
    "main_detection_50k_nd": main_detection_50k_nd_pd,
    "siam_ablation_5k": siam_ablation_5k_pd,
    "siam_preprocess_ablation_5k": siam_preprocess_ablation_5k_pd,
    "hap_dpp_retrieval": hap_dpp_retrieval_pd,
    "hap_dpp_detection_all": hap_dpp_detection_all_pd,
    "hap_dpp_detection_nd": hap_dpp_detection_nd_pd,
    "hap_dpp_vs_all_50k": hap_dpp_vs_all_50k_pd,
    "hap_dpp_vs_all_50k_nd": hap_dpp_vs_all_50k_nd_pd,
    "line_ablation_5k_retr": line_ablation_5k_retr_pd,
    "line_ablation_50k_retr": line_ablation_50k_retr_pd,
    "line_ablation_5k_det": line_ablation_5k_det_all_pd,
    "line_ablation_50k_det": line_ablation_50k_det_all_pd,
    "line_ablation_5k_det_nd": line_ablation_5k_det_nd_pd,
    "line_ablation_50k_det_nd": line_ablation_50k_det_nd_pd,
}

In [416]:
def paired_bootstrap_test(scores_a, scores_b, n_bootstrap=10000):
    rng = np.random.RandomState(42)
    scores_a = np.array(scores_a)
    scores_b = np.array(scores_b)
    n = len(scores_a)
    original_diff = np.mean(scores_a) - np.mean(scores_b)
    bootstrap_diffs = []
    for _ in range(n_bootstrap):
        indices = rng.choice(n, n, replace=True)
        bootstrap_a = scores_a[indices]
        bootstrap_b = scores_b[indices]
        bootstrap_diff = np.mean(bootstrap_a) - np.mean(bootstrap_b)
        bootstrap_diffs.append(bootstrap_diff)
    bootstrap_diffs = np.array(bootstrap_diffs)
    p_value = np.mean(bootstrap_diffs <= 0) if original_diff > 0 else np.mean(bootstrap_diffs >= 0)
    ci = np.percentile(bootstrap_diffs, [2.5, 97.5])
    return original_diff, ci, p_value

In [417]:
for experiment_name, df in all_dfs.items():
    print(f"Experiment: {experiment_name}")
    methods = df.columns
    for i in range(len(methods)):
        for j in range(i + 1, len(methods)):
            method_a = methods[i]
            method_b = methods[j]
            paired = df[[method_a, method_b]].dropna()
            scores_a = paired[method_a]
            scores_b = paired[method_b]
            if len(scores_a) == 0 or len(scores_b) == 0:
                continue
            diff, ci, p_value = paired_bootstrap_test(scores_a, scores_b)
            print(f"Comparison: {method_a} vs {method_b}")
            print(f"Mean difference: {diff:.4f}, 95% CI: [{ci[0]:.4f}, {ci[1]:.4f}], p-value: {p_value}")
    print("\n")

Experiment: main_retrieval_5k
Comparison: hap vs line
Mean difference: 0.0193, 95% CI: [-0.0083, 0.0494], p-value: 0.0907
Comparison: hap vs vvs
Mean difference: 0.1791, 95% CI: [0.1440, 0.2159], p-value: 0.0
Comparison: hap vs tmk
Mean difference: 0.7185, 95% CI: [0.6728, 0.7616], p-value: 0.0
Comparison: hap vs siamese
Mean difference: 0.8277, 95% CI: [0.7754, 0.8742], p-value: 0.0
Comparison: line vs vvs
Mean difference: 0.1599, 95% CI: [0.1080, 0.2099], p-value: 0.0
Comparison: line vs tmk
Mean difference: 0.6992, 95% CI: [0.6532, 0.7426], p-value: 0.0
Comparison: line vs siamese
Mean difference: 0.8052, 95% CI: [0.7555, 0.8528], p-value: 0.0
Comparison: vvs vs tmk
Mean difference: 0.5394, 95% CI: [0.4904, 0.5874], p-value: 0.0
Comparison: vvs vs siamese
Mean difference: 0.6540, 95% CI: [0.5916, 0.7127], p-value: 0.0
Comparison: tmk vs siamese
Mean difference: 0.1335, 95% CI: [0.0897, 0.1828], p-value: 0.0


Experiment: main_retrieval_50k
Comparison: hap vs line
Mean difference: -0